In [3]:
import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
from joblib import Parallel, delayed
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import adjusted_rand_score
from google.colab import files

warnings.filterwarnings("ignore")

# ==========================================
# 1. 極限高速化設定
# ==========================================
MAX_TRIALS = 100        # 最大試行回数を100に制限 (実質ほぼ20回で打ち切られます)
CHUNK_SIZE = 20         # 20回ごとにMCSEを判定
TARGET_MCSE = 0.015     # 許容誤差を少しだけ緩和 (論文レベルでも十分な精度)

DIMENSIONS = [10, 30, 50]
N_QUESTIONS = 20
# Nのプロット数を10から6に削減 (計算量を40%カット)
SAMPLE_SIZES = np.array([10**3, 10**4, 10**5, 10**6, 5*10**6, 10**7])

output_dir = "hypothesis4_spurious_clustering_ultra_fast"
os.makedirs(output_dir, exist_ok=True)

print(f"『虚構の同質性』極限高速スキャンを開始します...")
print(f"最大サンプル: {SAMPLE_SIZES[-1]:,} / 初期チャンク: {CHUNK_SIZE}回")

# ==========================================
# 2. 超高速化 DGP & K-means
# ==========================================
def simulate_kmeans_ari_ultra(D, N, M):
    K = 5

    cell_raw = np.random.uniform(0.01, 0.2, D)
    P_cell = cell_raw / np.sum(cell_raw)

    betas = 10 ** np.random.uniform(-1, 1.5, D)
    X_choice = np.random.uniform(0.01, 0.2, (D, M, K))

    exp_val = np.exp(betas[:, None, None] * X_choice)
    P_cond = exp_val / np.sum(exp_val, axis=2, keepdims=True)
    cum_P = np.cumsum(P_cond, axis=2)

    true_cells = np.random.choice(D, size=N, p=P_cell)

    Y = np.zeros((N, M), dtype=np.int8)
    rand_vals = np.random.rand(N, M, 1)

    for d in range(D):
        idx = np.where(true_cells == d)[0]
        if len(idx) > 0:
            Y[idx] = (rand_vals[idx] > cum_P[d]).sum(axis=2) + 1

    # ミニバッチをさらに小さくし、計算負荷を極小化
    batch_size = min(3000, N)
    kmeans = MiniBatchKMeans(n_clusters=D, batch_size=batch_size, n_init=1, max_iter=5)
    pred_clusters = kmeans.fit_predict(Y)

    return adjusted_rand_score(true_cells, pred_clusters)

# ==========================================
# 3. 動的早期打ち切りループ
# ==========================================
results_summary = []

for D in DIMENSIONS:
    print(f"\n--- カテゴリ次元数 D = {D} のスキャン ---")
    for N_samp in SAMPLE_SIZES:
        n_jobs = -1 if N_samp < 1000000 else 2

        trial_results = []
        mcse_ari = 1.0

        for chunk in range(MAX_TRIALS // CHUNK_SIZE):
            chunk_res = Parallel(n_jobs=n_jobs)(
                delayed(simulate_kmeans_ari_ultra)(D, N_samp, N_QUESTIONS) for _ in range(CHUNK_SIZE)
            )
            trial_results.extend(chunk_res)

            mean_ari = np.mean(trial_results)
            std_ari = np.std(trial_results) if len(trial_results) > 1 else 0
            mcse_ari = std_ari / np.sqrt(len(trial_results))

            if len(trial_results) >= CHUNK_SIZE and mcse_ari < TARGET_MCSE:
                break

        results_summary.append({
            "Dimension_D": D,
            "Sample_Size_N": N_samp,
            "Mean_ARI": mean_ari,
            "MCSE_ARI": mcse_ari,
            "Trials_Executed": len(trial_results)
        })
        print(f" N={N_samp:<10,} | ARI={mean_ari:.4f} | 試行回数={len(trial_results)} | MCSE={mcse_ari:.5f}")

df_res = pd.DataFrame(results_summary)
df_res.to_csv(f"{output_dir}/spurious_clustering_metrics_ultra.csv", index=False)

# ==========================================
# 4. 可視化・保存
# ==========================================
plt.style.use('seaborn-v0_8-whitegrid')
plt.figure(figsize=(10, 6))

colors = {10: 'tab:blue', 30: 'tab:green', 50: 'tab:red'}

for D in DIMENSIONS:
    subset = df_res[df_res["Dimension_D"] == D]
    plt.errorbar(subset["Sample_Size_N"], subset["Mean_ARI"],
                 yerr=1.96*subset["MCSE_ARI"], marker='o', color=colors[D],
                 capsize=4, linewidth=2, label=f'Dimension D={D}')

plt.axhline(y=0.5, color='gray', linestyle='--', label='Valid Capture Threshold')
plt.axhline(y=0.0, color='black', linestyle=':', label='Complete Noise')
plt.xscale('log')
plt.ylim(-0.05, 1.05)
plt.title("Collapse of 'Identical Statistical Population' in K-means (Ultra Fast)")
plt.xlabel("Sample Size N (log scale)")
plt.ylabel("Adjusted Rand Index")
plt.legend()
plt.tight_layout()
plt.savefig(f"{output_dir}/fig_spurious_homogeneity_ultra.png", dpi=300)
plt.close()

zip_filename = "hypothesis4_spurious_clustering_ultra.zip"
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files_list in os.walk(output_dir):
        for file in files_list:
            zipf.write(os.path.join(root, file), arcname=file)

print(f"\n極限高速検証が完了しました。結果を圧縮しました: {zip_filename}")
files.download(zip_filename)

『虚構の同質性』極限高速スキャンを開始します...
最大サンプル: 10,000,000 / 初期チャンク: 20回

--- カテゴリ次元数 D = 10 のスキャン ---
 N=1,000      | ARI=0.0297 | 試行回数=20 | MCSE=0.00487
 N=10,000     | ARI=0.0236 | 試行回数=20 | MCSE=0.00711
 N=100,000    | ARI=0.0475 | 試行回数=20 | MCSE=0.01094
 N=1,000,000  | ARI=0.0227 | 試行回数=20 | MCSE=0.00707
 N=5,000,000  | ARI=0.0565 | 試行回数=40 | MCSE=0.00993
 N=10,000,000 | ARI=0.0414 | 試行回数=20 | MCSE=0.00983

--- カテゴリ次元数 D = 30 のスキャン ---
 N=1,000      | ARI=0.0221 | 試行回数=20 | MCSE=0.00269
 N=10,000     | ARI=0.0223 | 試行回数=20 | MCSE=0.00323
 N=100,000    | ARI=0.0225 | 試行回数=20 | MCSE=0.00324
 N=1,000,000  | ARI=0.0232 | 試行回数=20 | MCSE=0.00313
 N=5,000,000  | ARI=0.0245 | 試行回数=20 | MCSE=0.00332
 N=10,000,000 | ARI=0.0298 | 試行回数=20 | MCSE=0.00363

--- カテゴリ次元数 D = 50 のスキャン ---
 N=1,000      | ARI=0.0112 | 試行回数=20 | MCSE=0.00145
 N=10,000     | ARI=0.0169 | 試行回数=20 | MCSE=0.00291
 N=100,000    | ARI=0.0179 | 試行回数=20 | MCSE=0.00237
 N=1,000,000  | ARI=0.0193 | 試行回数=20 | MCSE=0.00197
 N=5,000,000  | ARI

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>